In [ ]:
import json
import re
import math
from typing import List, Dict, Tuple
from collections import Counter

# ── The problem: 30 examples of a new "fraud_report" intent ─────────

NEW_INTENT = "fraud_report"
NEW_EXAMPLES = [
    "Someone made a purchase on my account that I didn't authorize",
    "I think my account has been hacked, there are charges I don't recognize",
    "There's fraudulent activity on my account",
    "I need to report unauthorized transactions",
    "Someone stole my identity and opened an account",
    # ... imagine 25 more like these
]

EXISTING_INTENTS = ["billing", "cancellation", "technical", "account_change", "general_inquiry"]


# ═══════════════════════════════════════════════════════════════════════
# APPROACH 1: Zero-shot — no examples needed, just a description
# ═══════════════════════════════════════════════════════════════════════
# Tradeoff: fastest to deploy, no training, but least accurate.
#           Works if the intent name is self-descriptive.

def zero_shot_classify(text: str, intents: List[str]) -> Dict:
    """Ask the LLM to classify with just intent names."""
    prompt = f"""Classify this customer message into exactly one intent.
Intents: {intents}
Message: "{text}"
Output JSON: {{"intent": "...", "confidence": "high/medium/low"}}"""

    raw = mock_llm_call(prompt)
    return json.loads(raw)


# ═══════════════════════════════════════════════════════════════════════
# APPROACH 2: Few-shot — give 3-5 examples per intent in the prompt
# ═══════════════════════════════════════════════════════════════════════
# Tradeoff: much better accuracy, but costs more tokens per call.
#           Limited by context window if many intents.

def few_shot_classify(text: str, examples_per_intent: Dict[str, List[str]]) -> Dict:
    """Include labeled examples in the prompt."""
    examples_block = ""
    for intent, examples in examples_per_intent.items():
        for ex in examples[:3]:  # 3 examples each to save tokens
            examples_block += f'  "{ex}" → {intent}\n'

    prompt = f"""Classify this customer message based on the examples below.

Examples:
{examples_block}
Message: "{text}"
Output JSON: {{"intent": "...", "confidence": "high/medium/low"}}"""

    raw = mock_llm_call(prompt)
    return json.loads(raw)


# ═══════════════════════════════════════════════════════════════════════
# APPROACH 3: Embedding similarity — no LLM needed at inference time
# ═══════════════════════════════════════════════════════════════════════
# Tradeoff: fastest at inference (<5ms), cheapest ($0), works offline.
#           Less flexible than LLM, accuracy depends on embedding quality.
#
# This is the approach Cresta would likely prefer for real-time hints
# because of the latency budget (<50ms).

def tfidf_embed(text: str, vocab_idf: Dict[str, float]) -> Dict[str, float]:
    """Simple TF-IDF vector as a stand-in for sentence-transformers."""
    words = re.findall(r'[a-z]+', text.lower())
    tf = Counter(words)
    total = len(words) if words else 1
    return {w: (c / total) * vocab_idf.get(w, 0) for w, c in tf.items()}


def cosine_sim(a: Dict, b: Dict) -> float:
    shared = set(a.keys()) & set(b.keys())
    dot = sum(a[k] * b[k] for k in shared)
    norm_a = math.sqrt(sum(v ** 2 for v in a.values()))
    norm_b = math.sqrt(sum(v ** 2 for v in b.values()))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


class EmbeddingSimilarityClassifier:
    """
    Stores embeddings of all labeled examples.
    Classifies by finding the most similar example.
    In production: swap tfidf_embed for sentence-transformers.
    """

    def __init__(self):
        self.examples: List[Tuple[str, Dict]] = []   # (intent, vector)
        self.vocab_idf: Dict[str, float] = {}

    def fit(self, labeled_data: Dict[str, List[str]]):
        """Index all labeled examples. Takes seconds, not hours."""
        # Build IDF from all examples
        all_texts = [t for examples in labeled_data.values() for t in examples]
        N = len(all_texts)
        df = Counter()
        for text in all_texts:
            for word in set(re.findall(r'[a-z]+', text.lower())):
                df[word] += 1
        self.vocab_idf = {w: math.log(N / c) for w, c in df.items()}

        # Store each example as (intent, vector)
        self.examples = []
        for intent, texts in labeled_data.items():
            for text in texts:
                vec = tfidf_embed(text, self.vocab_idf)
                self.examples.append((intent, vec))

    def predict(self, text: str, top_k: int = 5) -> Dict:
        """Find the k nearest examples, majority vote on intent."""
        query_vec = tfidf_embed(text, self.vocab_idf)

        # Score against all stored examples
        scored = []
        for intent, vec in self.examples:
            sim = cosine_sim(query_vec, vec)
            scored.append((intent, sim))
        scored.sort(key=lambda x: -x[1])

        # Majority vote from top_k
        top = scored[:top_k]
        votes = Counter(intent for intent, _ in top)
        winner = votes.most_common(1)[0][0]
        confidence = votes[winner] / top_k  # 5/5 = 1.0, 3/5 = 0.6

        return {
            "intent": winner,
            "confidence": round(confidence, 2),
            "top_match_score": round(top[0][1], 3),
        }


# ═══════════════════════════════════════════════════════════════════════
# Compare all three
# ═══════════════════════════════════════════════════════════════════════

def mock_llm_call(prompt: str) -> str:
    if "fraud" in prompt.lower() or "unauthorized" in prompt.lower() or "hacked" in prompt.lower():
        return json.dumps({"intent": "fraud_report", "confidence": "high"})
    elif "cancel" in prompt.lower():
        return json.dumps({"intent": "cancellation", "confidence": "high"})
    return json.dumps({"intent": "billing", "confidence": "medium"})


if __name__ == "__main__":
    test_messages = [
        "I see charges on my account that I didn't make, I think I've been hacked",
        "I want to cancel my service",
        "Why is my bill so high this month",
    ]

    # Approach 1: Zero-shot
    print("── Zero-shot (no examples) ──")
    all_intents = EXISTING_INTENTS + [NEW_INTENT]
    for msg in test_messages:
        result = zero_shot_classify(msg, all_intents)
        print(f"  '{msg[:50]}...' → {result['intent']}")

    # Approach 2: Few-shot
    print("\n── Few-shot (3 examples per intent) ──")
    examples = {
        "billing": ["My bill is too high", "I was charged twice", "Explain my charges"],
        "cancellation": ["I want to cancel", "Close my account", "I'm switching providers"],
        "fraud_report": NEW_EXAMPLES[:3],
    }
    for msg in test_messages:
        result = few_shot_classify(msg, examples)
        print(f"  '{msg[:50]}...' → {result['intent']}")

    # Approach 3: Embedding similarity
    print("\n── Embedding similarity (30 examples, no LLM at inference) ──")
    full_labeled = {
        "billing": ["My bill is too high", "I was charged twice", "Explain my charges",
                     "Why did my bill go up", "I need to dispute a charge"],
        "cancellation": ["I want to cancel", "Close my account", "I'm switching providers",
                         "How do I cancel", "Please terminate my service"],
        "technical": ["My internet is slow", "WiFi keeps dropping", "No connection",
                      "My speeds are terrible", "The app keeps crashing"],
        "fraud_report": NEW_EXAMPLES[:5],
    }
    clf = EmbeddingSimilarityClassifier()
    clf.fit(full_labeled)
    for msg in test_messages:
        result = clf.predict(msg)
        print(f"  '{msg[:50]}...' → {result['intent']} (conf={result['confidence']}, sim={result['top_match_score']})")

    # ── Tradeoff summary ──
    print("""
┌────────────────────┬──────────┬──────────┬──────────┬──────────┐
│ Approach           │ Accuracy │ Latency  │ Cost     │ Setup    │
├────────────────────┼──────────┼──────────┼──────────┼──────────┤
│ Zero-shot          │ Low-Med  │ 200ms    │ $$       │ Minutes  │
│ Few-shot           │ Med-High │ 300ms    │ $$$      │ Minutes  │
│ Embedding (TF-IDF) │ Medium   │ <5ms     │ $0       │ Minutes  │
│ Fine-tuned BERT    │ Highest  │ <10ms    │ $0       │ Hours    │
└────────────────────┴──────────┴──────────┴──────────┴──────────┘
    """)